<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1019/semantic_distance_layer_probe_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1019 v2 — Layer-wise changes in suppressed-concept representations

**Main question:** How does the semantic distance of a cue affect X-vs-Y decodability **before versus after reading the cue**? Does that relationship differ when concept X is explicitly suppressed?

**Design:** 8 fixed concept pairs × 12 prespecified cue sentences each × 2 instruction conditions (`neutral` and `suppression`) × all 24 transformer layers in Qwen2.5-0.5B-Instruct. A second, separate sentence encoder estimates semantic distance. Probe training uses multiple named and name-free concept descriptions; held-out descriptions are distinct from training examples. Each experimental input contains two explicit `STATE:` measurement markers. Readouts use the **same model forward pass**, leveraging the causal mask so the pre-cue marker cannot attend to the cue.

**Caveats:** These are *token-position differences in an autoregressive prompt*, not evidence of thinking, forgetting, temporal memory loss, or a causal mechanism. X remains written once in both instructions. Probe scores represent **X-vs-Y decodability** rather than absolute concept strength. The experiment is exploratory; 8 concept pairs are not representative of all language models or categories.

**Output location:** `GraduationThesis/1019/results_v2/<UTC-run-id>/` in Colab; optionally upload CSV/PNG/JSON to the same path in GitHub using Colab Secret `GITHUB_RESULTS_TOKEN`. A downloadable ZIP is created regardless of token availability.


## 0. Colab setup

Use **Runtime → Change runtime type → T4 GPU**. Avoid replacing Colab's PyTorch/CUDA build. Audio and vision packages are not required for text-only modeling.

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers>=4.54,<5" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.1" "matplotlib>=3.8" "tqdm>=4.65" "requests>=2.31"

In [ ]:
import os, io, re, json, sys, random, zipfile, base64, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import requests
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

SEED=42
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
EMBED_ID="sentence-transformers/all-MiniLM-L6-v2"
MODEL_LAYERS=list(range(1,25))       # every transformer layer: 1..24; 0 is token embedding
MAIN_LAYER=24                       # prespecified confirmatory-focus layer
AUC_CUTOFF=0.75                    # exploratory probe reliability screen
MIN_VALIDATED_FOR_LAYER_PLOT=5
CONDITIONS=["neutral","suppression"]
PROPOSED_LEVELS=["near","medium","far","unrelated"]
N_BOOT=2000
UPLOAD_TO_GITHUB=True
REPO="ryouy/GraduationThesis"
BRANCH="main"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
if DEVICE=="cpu":warnings.warn("No CUDA: select Colab T4 GPU for timely execution.")
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT=Path("/content/GraduationThesis") if Path("/content").is_dir() else Path.cwd()/"GraduationThesis"
OUT=ROOT/"1019"/"results_v2"/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
def save_csv(frame,name):
    frame.to_csv(OUT/name,index=False)
    print(f"Saved {name}: {len(frame)} rows")
def save_figure(name):
    plt.tight_layout();plt.savefig(OUT/name,dpi=190,bbox_inches="tight");plt.show();plt.close()
print("Device:",DEVICE,"| output:",OUT)

# Pin both auditable input banks to exact repository commits.
STIMULI_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
 "88579fa455f8f3f75085d395bcf30a313ba400dc/1019/stimuli_v2.csv")
DESCRIPTION_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
 "a6e058ed05c5a43e28cf46bdecd10eb54edcf19d/1019/probe_description_bank_v2.json")
def remote_bytes(url):
    r=requests.get(url,timeout=60);r.raise_for_status()
    return r.content
stimuli_bytes=remote_bytes(STIMULI_URL)
descriptions_bytes=remote_bytes(DESCRIPTION_URL)
cues=pd.read_csv(io.BytesIO(stimuli_bytes))
desc=json.loads(descriptions_bytes.decode("utf-8"))
CONCEPTS=sorted(cues.concept.unique().tolist())
assert len(CONCEPTS)==8 and len(cues)==96
assert set(cues.proposed_level)==set(PROPOSED_LEVELS)
assert cues.groupby(["concept","proposed_level"]).size().eq(3).all()
assert set(CONCEPTS)==set(desc)
def contains_word(s,w):
    return bool(re.search(r"(?<![A-Za-z])"+re.escape(w)+r"(?![A-Za-z])",s,re.I))
for row in cues.itertuples():
    assert not contains_word(row.cue_text,row.concept)
    assert not contains_word(row.cue_text,row.replacement)
for x,d in desc.items():
    for k in ["x_train","x_test","y_train","y_test"]:
        assert len(d[k])==3 and all(not contains_word(v,x) and
            not contains_word(v,cues.loc[cues.concept==x,"replacement"].iloc[0]) for v in d[k])
# Save the exact bank of prewritten inputs with every run.
(OUT/"stimuli_v2.csv").write_bytes(stimuli_bytes)
(OUT/"probe_description_bank_v2.json").write_bytes(descriptions_bytes)
INPUT_HASHES={"stimuli_sha256":hashlib.sha256(stimuli_bytes).hexdigest(),
              "descriptions_sha256":hashlib.sha256(descriptions_bytes).hexdigest()}
print("Input bank validated:",len(cues),"cues, 8 pairs. SHA256:",INPUT_HASHES)
display(cues.groupby(["concept","proposed_level"]).size().unstack())

## 1. Independent meaning-distance measurement

MiniLM estimates 1 − cosine between the cue and two short target-name sentences. Target distances are not evaluated using Qwen's hidden states or the learned X/Y probes. **Prewritten labels are audited, not assumed correct.** For descriptive quartile plots, sort the 12 cues per concept by measured distance into four equal groups of three; the continuous measured distance is the actual predictor. These are *embedding-based*, not human-validated distance categories.

In [ ]:
e_tok=AutoTokenizer.from_pretrained(EMBED_ID,trust_remote_code=False)
e_model=AutoModel.from_pretrained(EMBED_ID,trust_remote_code=False).cpu().eval()
def embed_sentences(texts,batch_size=32):
    result=[]
    for start in tqdm(range(0,len(texts),batch_size),desc="Independent semantic embeddings"):
        b=e_tok(texts[start:start+batch_size],padding=True,truncation=True,
                max_length=128,return_tensors="pt")
        with torch.inference_mode():
            h=e_model(**b).last_hidden_state
            mask=b.attention_mask.unsqueeze(-1)
            pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
            result.append(F.normalize(pooled.float(),dim=-1).cpu().numpy())
    return np.concatenate(result,axis=0)
anchors=[f"This text is about {x}." for x in CONCEPTS]+[
         f"The main topic is {x}." for x in CONCEPTS]
embedding=embed_sentences(anchors+cues.cue_text.tolist())
centers={}
for i,x in enumerate(CONCEPTS):
    center=(embedding[i]+embedding[i+len(CONCEPTS)])/2
    centers[x]=center/np.linalg.norm(center)
cues["semantic_similarity"]=[float(centers[r.concept]@embedding[2*len(CONCEPTS)+i])
                             for i,r in cues.reset_index(drop=True).iterrows()]
cues["semantic_distance"]=1-cues.semantic_similarity
# Within-concept empirical quartiles, made without seeing Qwen or probe outcomes.
cues=cues.sort_values(["concept","semantic_distance","cue_id"]).reset_index(drop=True)
cues["distance_rank"]=cues.groupby("concept").cumcount()+1
cues["measured_bin"]=cues.distance_rank.map(lambda rank:PROPOSED_LEVELS[(rank-1)//3])
cues["word_count"]=cues.cue_text.str.split().str.len()
audit=cues.groupby(["concept","proposed_level"],as_index=False).semantic_distance.mean()
audit_wide=audit.pivot(index="concept",columns="proposed_level",values="semantic_distance").reindex(columns=PROPOSED_LEVELS)
audit_wide["proposed_order_confirmed"]=np.all(np.diff(audit_wide[PROPOSED_LEVELS].to_numpy(),axis=1)>0,axis=1)
save_csv(cues,"01_stimulus_distances_and_ranks.csv")
save_csv(audit_wide.reset_index(),"02_proposed_distance_audit.csv")
display(audit_wide.round(3))
print("Proposed four-group order holds for",int(audit_wide.proposed_order_confirmed.sum()),"/8 concepts.")
print("Measured bins (quartiles) are descriptive and may reassign cues from their original categories.")
del e_model,e_tok,embedding
if torch.cuda.is_available():torch.cuda.empty_cache()